# Summary

Search the Bedrock Knowledge base

In [1]:
import os, sys
import pandas as pd
import re
import json
import time

# AWS Python
import boto3

utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")


## Construct a AWS Bedrock Retrieval and Generate class object

In [2]:
class BedrockKBRetriever:
    """
    Class to demonstrate AWS Bedrock Knowledge Base retrieve and retrieve and generate results.
    """

    def __init__(self,
                 aws_profile: str,
                 aws_region: str,
                 kb_id: str):

        self.aws_profile = aws_profile
        self.aws_region = aws_region
        self.kb_id = kb_id

        # Retrieval configuration
        self.num_srch_res = 5
        self.s3_bucket = "ashoka-search-tests"
        self.s3_doc_loc = "s3://{}/".format(self.s3_bucket)

        # Retrieval and Generation configuration
        self.aws_account_id = os.environ["AWS_ACCOUNT_ID"]
        # self.inference_model = "amazon.nova-lite-v1:0" #?
        # self.inference_model = "anthropic.claude-sonnet-4-6" # Requires Anthropic request form
        # self.inference_model = "google.gemma-3-4b-it" # Not available
        self.inference_model = "meta.llama4-scout-17b-instruct-v1:0"
        self.model_arn = "arn:aws:bedrock:{}:{}:inference-profile/us.{}".format(self.aws_region,
                                                                                self.aws_account_id,
                                                                                self.inference_model)
        self.search_sleep_time = 5 # pause time between queries
        self.ai_sleep_time = 5 # pause time between queries

        # Outputs
        self.df_ret_res = pd.DataFrame()
        self.df_query_finds = pd.DataFrame()
        self.df_rag_res = pd.DataFrame()
        self.bedrock_models = pd.DataFrame() # AWS foundation models - requires get_bedrock_foundational_models method

        # Establish an AWS client
        self.set_aws_client()

    def set_aws_client(self):
        """
        Create a AWS client
        :return:
        """

        # Initialize Bedrock Agent Runtime client for querying
        session = boto3.Session(profile_name=self.aws_profile)

        # Bedrock client
        self.br_rt = session.client('bedrock-agent-runtime',
                                    region_name=self.aws_region)

        # S3 client
        self.s3 = session.client('s3',
                                 region_name=self.aws_region)

    def get_bedrock_foundational_models(self):
        """
        Get a list of foundation models available through Bedrock
        :return:
        """

        # Initialize a bedrock client
        session = boto3.Session(profile_name=self.aws_profile)
        self.br = session.client('bedrock', region_name=self.aws_region)

        response = self.br.list_foundation_models()
        self.response_models = response

        models_rows = []
        for model in response['modelSummaries']:
            models_rows.append(dict(model_name=model['modelName'],
                                    model_id=model['modelId'],
                                    model_provider=model['providerName'],
                                    input_modalities=model['inputModalities'],
                                    output_modalities=model['outputModalities'],
                                    infer_types=model['inferenceTypesSupported'],
                                    model_lifecycle=model['modelLifecycle']
                                    )
                               )

        self.bedrock_models = pd.DataFrame(data=models_rows)
        self.bedrock_models = self.bedrock_models.sort_values(by="model_provider")
        self.bedrock_models = self.bedrock_models.reset_index(drop=True)

    def retrieve_query_results(self,
                               queries: list,
                               source_types: list = []):
        """
        Return AWS Bedrock search results for a list of queries. Filtering before searching can be
        applied by passing source type values in the source_types input parameter.
        :return:
        """

        # Set retrieval configurations
        if len(source_types) == 0:
            ret_config = {'vectorSearchConfiguration': {
                                                        'numberOfResults': self.num_srch_res
                                                        }
                         }

        if len(source_types) > 0:
            ret_config = {'vectorSearchConfiguration': {
                                                        'numberOfResults': self.num_srch_res,
                                                        'filter': {
                                                            'in': {
                                                                'key': 'source_type',
                                                                'value': source_types
                                                            }
                                                        }
                                                        }
                         }

        search_results = []
        for query in queries:

            # Set retrieval configurations
            ret_query_param = {'text': query
                               }

            response = self.br_rt.retrieve(knowledgeBaseId=self.kb_id,
                                           retrievalQuery=ret_query_param,
                                           retrievalConfiguration=ret_config
                                           )

            for result in response["retrievalResults"]:

                # Get list of passages
                regex_pattern = r"Passage\s#(\d+)"

                passages = re.findall(regex_pattern, result['content']['text'], flags=re.DOTALL)

                # Clean up whitespace from the results
                passages = [p.strip() for p in passages]

                # Get metadata
                s3_loc=result['location']['s3Location']['uri']
                doc_name = s3_loc.replace(self.s3_doc_loc, "")
                key = "{}.metadata.json".format(doc_name)

                # Read the file from S3
                response_docread = self.s3.get_object(Bucket=self.s3_bucket,
                                                      Key=key)
                content = response_docread['Body'].read().decode('utf-8')

                # Parse JSON into dictionary
                metadata_dict = json.loads(content)
                source_type = metadata_dict["metadataAttributes"]["source_type"]

                res_dict = dict(query=query,
                                search_res=result['content']['text'],
                                passages=passages,
                                score=result['score'],
                                s3_loc=result['location']['s3Location']['uri'],
                                source_type=source_type,
                                doc_metadata=metadata_dict
                                )

                search_results.append(res_dict)

                time.sleep(self.search_sleep_time)

        # Put the detailed retrieval results into a dataframe
        self.df_ret_res = pd.DataFrame(search_results)

        # Aggregate results by query
        self.aggregate_retrieval_results()

    def aggregate_retrieval_results(self):
        """
        Aggregate retrieval results by query
        :param queries:
        :return:
        """

        q_rows = []
        for query in self.df_ret_res["query"].unique():

            mask = self.df_ret_res["query"] == query

            doc_value_cnts = self.df_ret_res[mask]["s3_loc"].value_counts()

            # Get passages
            pass_dict = {}
            for idx in self.df_ret_res[mask].index:
                doc_filename = self.df_ret_res.loc[idx, "s3_loc"].replace(self.s3_doc_loc, "")

                if doc_filename in pass_dict:
                    pass_dict[doc_filename] = pass_dict[doc_filename] + self.df_ret_res.loc[idx, "passages"]
                else:
                    pass_dict[doc_filename] = self.df_ret_res.loc[idx, "passages"]

            for df in pass_dict:
                pass_dict[df] = list(set(pass_dict[df]))
                pass_dict[df].sort()

            doc_dict = {}
            for s3_loc_idx in doc_value_cnts.index:
                ikey = s3_loc_idx.replace(self.s3_doc_loc, "")
                doc_dict[ikey] = int(doc_value_cnts[s3_loc_idx])

            doc_count = len(doc_dict)

            max_score = self.df_ret_res[mask]["score"].max()

            source_types = self.df_ret_res[mask]["source_type"].unique().tolist()

            q_rows.append(dict(query=query,
                               doc_count=doc_count,
                               max_score=max_score,
                               doc_scores=doc_dict,
                               passages=pass_dict,
                               source_types=source_types
                               )
                          )

        self.df_query_finds = pd.DataFrame(q_rows)

    def retrieve_and_generate_results(self,
                                      queries: list,
                                      source_types: list = []):
        """
        Retrieve AWS Bedrock search results and for a list of queries and generate AI responses summarizing the findings. Filtering before searching can be
        applied by passing source type values in the source_types input parameter.
        :return:
        """

        # Set retrieval configurations
        if len(source_types) == 0:
            ret_gen_config = {'type': 'KNOWLEDGE_BASE',
                              'knowledgeBaseConfiguration': {'knowledgeBaseId': self.kb_id,
                                                             'modelArn': self.model_arn
                                                             }
                              }

        if len(source_types) > 0:
              ret_gen_config = {'type': 'KNOWLEDGE_BASE',
                              'knowledgeBaseConfiguration': {'knowledgeBaseId': self.kb_id,
                                                             'modelArn': self.model_arn,
                                                             'retrievalConfiguration': {
                                                                'vectorSearchConfiguration': {
                                                                    'numberOfResults': self.num_srch_res,
                                                                    'filter': {
                                                                        'in': {
                                                                            'key': 'source_type',
                                                                            'value':  source_types
                                                                        }
                                                                    }
                                                                }
                                                             }
                                                             }
                                }

        search_results = []
        for query in queries:

            rag_response = self.br_rt.retrieve_and_generate(input={'text': query},
                                                            retrieveAndGenerateConfiguration=ret_gen_config
                                                            )

            rag_answer = rag_response['output']['text']

            citations_list = []
            for citation in rag_response['citations']:
                for reference in citation['retrievedReferences']:
                    if 'location' in reference:
                        if 's3Location' in reference['location']:
                            citations_list.append(reference['location']['s3Location']['uri'])

            search_results.append(dict(query=query,
                                       rag_answer=rag_answer,
                                       rag_citations=citations_list)
                                    )

            time.sleep(self.ai_sleep_time)

        self.df_rag_res = pd.DataFrame(search_results)


## Read test data

In [3]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_mpqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, multi_pas_qs))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))


In [5]:
# df_docs


## Get available foundational models

In [34]:
# Set up some parameters
aws_profile = "ns-admin"
aws_region = 'us-east-2'
kb_id = "DBSECKYXJV"

# Set up retriever object
srch_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
srch_analyzer.get_bedrock_foundational_models()


In [36]:
srch_analyzer.bedrock_models.loc[[20]].T
srch_analyzer.bedrock_models.loc[[31]].T
# srch_analyzer.bedrock_models

# srch_analyzer.response_models



,31
model_name,Llama 4 Scout 17B Instruct
model_id,meta.llama4-scout-17b-instruct-v1:0
model_provider,Meta
input_modalities,"[TEXT, IMAGE]"
output_modalities,[TEXT]
infer_types,[INFERENCE_PROFILE]
model_lifecycle,"{'status': 'ACTIVE', 'startOfLifeTime': 2025-0..."


## Retrieve Search Results

In [6]:
# Set up some parameters
aws_profile = "ns-admin"
aws_region = 'us-east-2'
kb_id = "DBSECKYXJV"

# Set up retriever object
srch_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
queries = df_spqs["question"].tolist()

# Search documents for text related to queries without filtering
srch_analyzer.retrieve_query_results(queries=queries)

# Search documents for text related to queries with filtering
# srch_analyzer.retrieve_query_results(queries=queries,
#                                      source_types=["gaming"])

In [11]:

# srch_analyzer.df_ret_res
srch_analyzer.df_query_finds


,query,doc_count,max_score,doc_scores,passages,source_types
0,What do keybullet kin drop?,1,0.637887,{'doc_0.txt': 5},"{'doc_0.txt': ['1', '2', '26', '27', '28', '29...",[gaming]
1,What kind of gun does the bandana bullet kin use?,1,0.591255,{'doc_0.txt': 5},"{'doc_0.txt': ['1', '10', '11', '12', '13', '1...",[gaming]
2,What do the giants look like?,2,0.371861,"{'doc_15.txt': 3, 'doc_0.txt': 2}","{'doc_15.txt': [], 'doc_0.txt': ['10', '11', '...","[entertainment, gaming]"
3,What happens on day 2?,3,0.381578,"{'doc_16.txt': 2, 'doc_18.txt': 2, 'doc_9.txt'...","{'doc_16.txt': [], 'doc_18.txt': ['19', '20', ...","[gaming, entertainment]"
4,What were the requirements for the project?,5,0.377493,"{'doc_6.txt': 1, 'doc_2.txt': 1, 'doc_5.txt': ...","{'doc_6.txt': ['57', '58', '59', '60', '61'], ...","[government, data_science, entertainment, gaming]"
5,What data did was used to test the prototype?,1,0.440606,{'doc_2.txt': 5},"{'doc_2.txt': ['17', '18', '19', '20', '21', '...",[data_science]
6,How do the data storage options compare?,4,0.420020,"{'doc_2.txt': 2, 'doc_3.txt': 1, 'doc_16.txt':...","{'doc_3.txt': ['36'], 'doc_16.txt': [], 'doc_2...","[data_science, gaming]"
7,When was UTF-8 support added for European lang...,3,0.384851,"{'doc_3.txt': 2, 'doc_16.txt': 2, 'doc_11.txt'...","{'doc_11.txt': ['75', '76', '77'], 'doc_3.txt'...","[data_science, gaming]"
8,How do I make a button?,1,0.422285,{'doc_4.txt': 5},"{'doc_4.txt': ['100', '101', '102', '103', '50...",[recipes]
9,When might I use caching?,3,0.455322,"{'doc_4.txt': 2, 'doc_19.txt': 2, 'doc_11.txt'...","{'doc_4.txt': ['1', '103', '104', '105', '106'...","[recipes, data_science]"


## Retrieve and Generate AI Responses using RAG

In [7]:
# Set up some parameters
aws_profile = "ns-admin"
aws_region = 'us-east-2'
kb_id = "DBSECKYXJV"

# Set up retriever object
rag_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
queries = df_spqs["question"].tolist()

# Retrieve RAG responses for text related to queries without filtering
rag_analyzer.retrieve_and_generate_results(queries=queries)

# Search RAG responses for text related to queries with filtering
# rag_analyzer.retrieve_and_generate_results(queries=queries,
#                                            source_types=["gaming"])


In [8]:

rag_analyzer.df_rag_res



,query,rag_answer,rag_citations
0,What do keybullet kin drop?,"Keybullet Kin drop a key upon death. However, ...","[s3://ashoka-search-tests/doc_0.txt, s3://asho..."
1,What kind of gun does the bandana bullet kin use?,The Bandana Bullet Kin wields a Machine Pistol.,[s3://ashoka-search-tests/doc_0.txt]
2,What do the giants look like?,The search results do not provide a descriptio...,"[s3://ashoka-search-tests/doc_15.txt, s3://ash..."
3,What happens on day 2?,"Sorry, I am unable to assist you with this req...",[]
4,What were the requirements for the project?,"The requirements for the project, which was to...",[s3://ashoka-search-tests/doc_2.txt]
5,What data did was used to test the prototype?,The prototype was tested using Wikipedia pages...,[s3://ashoka-search-tests/doc_2.txt]
6,How do the data storage options compare?,The data storage options include several combi...,[s3://ashoka-search-tests/doc_3.txt]
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,[s3://ashoka-search-tests/doc_3.txt]
8,How do I make a button?,"To create a button, you can use the `mo.ui.but...","[s3://ashoka-search-tests/doc_4.txt, s3://asho..."
9,When might I use caching?,You might use caching when you have expensive ...,"[s3://ashoka-search-tests/doc_4.txt, s3://asho..."


In [9]:
df_rag_res_ans = rag_analyzer.df_rag_res.copy(deep=True)

df_rag_res_ans["answer"] = ""
for idx in df_rag_res_ans.index:
    if df_rag_res_ans.loc[idx, "query"] == df_spqs.loc[idx, "question"]:
        df_rag_res_ans.loc[idx, "answer"] = df_spqs.loc[idx, "answer"]
    else:
        print("query problem: {}".format(idx))

# df_spqs
df_rag_res_ans



,query,rag_answer,rag_citations,answer
0,What do keybullet kin drop?,"Keybullet Kin drop a key upon death. However, ...","[s3://ashoka-search-tests/doc_0.txt, s3://asho...",Keybullet kin drop a key upon death.
1,What kind of gun does the bandana bullet kin use?,The Bandana Bullet Kin wields a Machine Pistol.,[s3://ashoka-search-tests/doc_0.txt],The bandana bullet kin wields a machine pistol.
2,What do the giants look like?,The search results do not provide a descriptio...,"[s3://ashoka-search-tests/doc_15.txt, s3://ash...","One giant is burly, grey-skinned, and 20 feet ..."
3,What happens on day 2?,"Sorry, I am unable to assist you with this req...",[],"After a few miles of winding tunnel, you emerg..."
4,What were the requirements for the project?,"The requirements for the project, which was to...",[s3://ashoka-search-tests/doc_2.txt],The tool had the following requirements:\n- Ch...
5,What data did was used to test the prototype?,The prototype was tested using Wikipedia pages...,[s3://ashoka-search-tests/doc_2.txt],Grace Hopper's Wikipedia page and Alan Turing'...
6,How do the data storage options compare?,The data storage options include several combi...,[s3://ashoka-search-tests/doc_3.txt],For fast start: use SQLite3 and ChromaDB (File...
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,[s3://ashoka-search-tests/doc_3.txt],UTF-8 support was added for European languages...
8,How do I make a button?,"To create a button, you can use the `mo.ui.but...","[s3://ashoka-search-tests/doc_4.txt, s3://asho...",import marimo as mo\n\nbutton = mo.ui.run_butt...
9,When might I use caching?,You might use caching when you have expensive ...,"[s3://ashoka-search-tests/doc_4.txt, s3://asho...","You might use caching when, for example, your ..."
